## Comparing data IBtracks and TC Forecast 

In [ ]:
from climada_petals.hazard import TCForecast
import logging
import datetime
import os
import numpy as np
import geopandas as gpd

In [ ]:
os.chdir('/cluster/project/climate/gespejo/')

In [ ]:
from datetime import datetime, timedelta
import logging
import os

logging.getLogger('climada.hazard.tc_tracks_forecast').setLevel(logging.WARNING)

def fetch_multiple_ecmwf_data(base_folder, start_date, end_date, event_name):
    current_date = start_date
    forecast_variables = {}  # Dictionary to store forecast variables

    while current_date <= end_date:
        for utc_hour in [0,18]:  # Only check 0 UTC and 12 UTC
            date_folder = current_date.replace(hour=utc_hour, minute=0, second=0).strftime("%Y%m%d%H%M%S")
            folder_path = os.path.join(base_folder, date_folder)

            if os.path.exists(folder_path):
                forecast = TCForecast()
                files_to_fetch = []
                for root, dirs, files in os.walk(folder_path):
                    for file in files:
                        if event_name in file:
                            files_to_fetch.append(os.path.join(root, file))

                if files_to_fetch:
                    print(f"Fetching data for {date_folder}")
                    forecast.fetch_ecmwf(files=files_to_fetch)
                    variable_name = f"{event_name}_{date_folder}"
                    forecast_variables[variable_name] = forecast
                else:
                    print(f"No data found for {date_folder}")
            else:
                print(f"Folder not found: {date_folder}")

        current_date += timedelta(days=1)

    return forecast_variables

In [ ]:
# Set your parameters here
base_folder = "/cluster/project/climate/gespejo/data/TC_0012"
start_date = datetime(2023,2,16,18,0)
end_date = datetime(2023, 2,21,18, 0)
event_name = "FREDDY"

# Fetch data and store forecast variables
forecast_variables = fetch_multiple_ecmwf_data(base_folder, start_date, end_date, event_name)

In [ ]:
# construct centroids among Mozambique and La Reunion 
from climada.hazard import Centroids
min_lat, max_lat, min_lon, max_lon = -27,-10, 42, 52
cent = Centroids.from_pnt_bounds((min_lon, min_lat, max_lon, max_lat), res=0.12)
cent.check()
#cent.plot();

In [ ]:
from climada.hazard.trop_cyclone import TropCyclone 
from datetime import datetime

# Create an empty dictionary to store TropCyclone objects
tc_objects = {}

# Create dictionaries instead of lists to store TropCyclone names and ensemble variables
tc_names = {}
ens_variables = {}

for variable_name, forecast in forecast_variables.items():
    if f"{event_name}" in variable_name:
        ensemble_variable_name = f"ens_{variable_name}"
        ensemble_forecast = forecast.subset({'is_ensemble': True})
        ensemble_forecast.equal_timestep(time_step_h=0.5)
        globals()[variable_name] = forecast
        globals()[ensemble_variable_name] = ensemble_forecast

        print(f"Variable {ensemble_variable_name} created as the ensemble of {variable_name}")
        
        # Extract the date and time information from the variable name
        date_time_str = variable_name.split("_", 2)[-1]
        date_time_obj = datetime.strptime(date_time_str, "%Y%m%d%H%M%S")

        # Create a TropCyclone object from ensemble forecast data
        tc_object_name = f"{event_name}_{date_time_obj.strftime('%Y%m%d%H%M%S')}"
        
        # Assign name and ensemble variable to respective keys
        timestamp_key = f"{event_name}_{date_time_obj.strftime('%Y%m%d%H%M%S')}"
        tc_names[timestamp_key] = TropCyclone.from_tracks(ensemble_forecast, centroids=cent)

        # Assign ensemble variable name to the dict
        ens_variables[f"ens_{timestamp_key}"] = ensemble_forecast
        print(f"Variable ens_{timestamp_key} added to ens_variables")

In [ ]:
import copy

# Create a deep copy of tc_names to preserve the original names
tc_names_088 = copy.deepcopy(tc_names)

# Create an empty list to store the new variable names
new_tc_names = []

for tc_name_copy in tc_names_088:
    # Get the reference to the copied TropCyclone object
    tc_088_copy = copy.deepcopy(globals()[tc_name_copy])
    
    # Calculate the new intensity data
    new_data = tc_088_copy.intensity.data * 0.88
    
    # Update the intensity data of the copied TropCyclone object with the new data
    tc_088_copy.intensity.data = new_data

    # Create a new variable with the 'new_' prefix for the copy
    new_variable = 'new_' + tc_name_copy
    
    # Store a reference to the updated copied TropCyclone object in the new variable
    globals()[new_variable] = tc_088_copy

    # Append the new variable name to the list
    new_tc_names.append(new_variable)

# Now, new_tc_names contains variable names with reference to the copied and modified TropCyclone objects

In [ ]:
## ## EXTRACTING THE DATA OF THE ENSEMBLE MEMBERS IN A DATAFRAME PER DAY

In [ ]:
### EXTRACTING THE DATA OF THE ENSEMBLE MEMBERS IN A DATAFRAME 
import pandas as pd

# Initialize an empty dataframe to store the data
ens_members_20230217180000 = pd.DataFrame(columns=['Ensemble_Member', 'Time', 'Lon', 'Lat', 'Max_Sustained_Wind'])

# Loop through ensemble members from 1 to 51
for ensemble_member in range(1, 52):
    ens_member_name = f'ens_FREDDY_20230217180000'
    if ens_member_name in globals():
        ens_member_subset = globals()[ens_member_name].subset({'ensemble_number': ensemble_member})
        time_data = ens_member_subset.get_track('07S').time
        lon_data = ens_member_subset.get_track('07S').lon
        lat_data = ens_member_subset.get_track('07S').lat
        mswind_data = ens_member_subset.get_track('07S').max_sustained_wind

        # Create a temporary dataframe for the current ensemble member
        temp_df = pd.DataFrame({
            'Ensemble_Member': [ensemble_member] * len(time_data),
            'Time': time_data,
            'Lon': lon_data,
            'Lat': lat_data,
            'Max_Sustained_Wind': mswind_data
        })

        # Append the data for the current ensemble member to the main dataframe
        ens_members_20230217180000 = pd.concat([ens_members_20230217180000, temp_df], ignore_index=True)
    else:
        print(f"Ensemble member {ensemble_member} not found in globals.")

In [ ]:
#ens_members_20230217180000.to_excel('D:/UNIVERSITY_OF_BERN/MASTER_THESIS/dataframes_excel/ens_members_20230217180000_30min.xlsx', index=False)